In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

import numpy as np

import sys
import os
sys.path.append("../..")
from dataloader import get_dataset

from network import SinglePhaseBinaryDinoClassifier

In [4]:
import torch
def get_model(path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SinglePhaseBinaryDinoClassifier(input_dim=1025).to(device)
    model.load_state_dict(torch.load(path))
    model.eval()
    return model

In [9]:
one_model = get_model("/code/jjiang23/pathml/aim2_balance/models/5PhaseEnsemble/results/train_set_phase_one_v3/best_model.pth")
two_model = get_model("/code/jjiang23/pathml/aim2_balance/models/5PhaseEnsemble/results/train_set_phase_two_v3/best_model.pth")
three_model = get_model("/code/jjiang23/pathml/aim2_balance/models/5PhaseEnsemble/results/train_set_phase_three_v3/best_model.pth")
four_model = get_model("/code/jjiang23/pathml/aim2_balance/models/5PhaseEnsemble/results/train_set_phase_four_v3/best_model.pth")
nonphase_model = get_model("/code/jjiang23/pathml/aim2_balance/models/5PhaseEnsemble/results/train_set_nonphase_v3/best_model.pth")

/tmp/ipykernel_276480/3378701463.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(path))


In [11]:
#load dataset
features, labels, fps_list = get_dataset("/code/jjiang23/pathml/aim2_balance/processed_files/train_set.txt", "dinov3_features", frame_aware=True)

In [18]:
def predict_phase(model, vid_features):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    features = torch.tensor(vid_features, dtype=torch.float32).to(device)
    with torch.no_grad():
        outputs = model(features)
        outputs = outputs.squeeze(1)  # Added: ensure consistent dimensions
        predicted = torch.sigmoid(outputs).float()  # Fixed: use sigmoid for binary classification
    return predicted.cpu().numpy()

In [20]:
phase_one_probs = [predict_phase(one_model, vid_features) for vid_features in features]
phase_two_probs = [predict_phase(two_model, vid_features) for vid_features in features]
phase_three_probs = [predict_phase(three_model, vid_features) for vid_features in features]
phase_four_probs = [predict_phase(four_model, vid_features) for vid_features in features]
nonphase_probs = [predict_phase(nonphase_model, vid_features) for vid_features in features]

/tmp/ipykernel_276480/521293248.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  features = torch.tensor(vid_features, dtype=torch.float32).to(device)


In [33]:
vid_train = []
for i in range(len(features)):
    vid_train.append(np.array(list(zip(phase_one_probs[i], phase_two_probs[i], phase_three_probs[i], phase_four_probs[i], nonphase_probs[i]))))

In [35]:
len(vid_train), len(vid_train[0][1]), len(labels)

(104, 5, 104)

In [37]:
cat_vid_train_X = np.concatenate(vid_train, axis=0)
cat_vid_train_y = np.concatenate(labels, axis=0)

In [39]:
#set up holdout set
holdout_features, holdout_labels, holdout_fps_list = get_dataset("/code/jjiang23/pathml/aim2_balance/processed_files/holdout_set.txt", "dinov3_features", frame_aware=True)

In [40]:
holdout_phase_one_probs = [predict_phase(one_model, vid_features) for vid_features in holdout_features]
holdout_phase_two_probs = [predict_phase(two_model, vid_features) for vid_features in holdout_features]
holdout_phase_three_probs = [predict_phase(three_model, vid_features) for vid_features in holdout_features]
holdout_phase_four_probs = [predict_phase(four_model, vid_features) for vid_features in holdout_features]
holdout_nonphase_probs = [predict_phase(nonphase_model, vid_features) for vid_features in holdout_features]
vid_holdout = []
for i in range(len(holdout_features)):
    vid_holdout.append(np.array(list(zip(holdout_phase_one_probs[i], holdout_phase_two_probs[i], holdout_phase_three_probs[i], holdout_phase_four_probs[i], holdout_nonphase_probs[i]))))   
cat_vid_holdout_X = np.concatenate(vid_holdout, axis=0)
cat_vid_holdout_y = np.concatenate(holdout_labels, axis=0)

/tmp/ipykernel_276480/521293248.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  features = torch.tensor(vid_features, dtype=torch.float32).to(device)


In [38]:
rf = RandomForestClassifier(
    n_estimators=200,       # number of trees
    max_depth=None,         # let trees grow deep
    random_state=42,
    class_weight="balanced" # helps if classes are imbalanced
)
rf.fit(cat_vid_train_X, cat_vid_train_y)

# Evaluate
# y_pred = rf.predict(X_test)
# print(classification_report(y_test, y_pred))

RandomForestClassifier(class_weight='balanced', n_estimators=200,
                       random_state=42)

In [41]:
y_pred = rf.predict(cat_vid_holdout_X)
print(classification_report(cat_vid_holdout_y, y_pred))

              precision    recall  f1-score   support

           0       0.59      0.27      0.37      7621
           1       0.39      0.09      0.14      7069
           2       0.27      0.08      0.12      6148
           3       0.59      0.38      0.46      4934
           4       0.68      0.92      0.78     44174

    accuracy                           0.65     69946
   macro avg       0.50      0.35      0.37     69946
weighted avg       0.60      0.65      0.59     69946

